In [7]:
# from gulps.synthesis_plugin import GulpsSynthesisPlugin
import numpy as np
from monodromy.render import _plot_coverage_set
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.circuit.library import (
    CXGate,
    RZXGate,
    UGate,
    UnitaryGate,
    XXPlusYYGate,
    iSwapGate,
)
from qiskit.circuit.random import random_circuit
from qiskit.quantum_info import Operator, average_gate_fidelity
from qiskit.quantum_info.random import random_unitary
from qiskit.transpiler import (
    InstructionProperties,
    PassManager,
    Target,
    generate_preset_pass_manager,
)
from qiskit.transpiler.passes import Optimize1qGatesDecomposition
from tqdm import tqdm
from weylchamber import c1c2c3

from gulps.gulps_synthesis import GulpsDecomposer
from gulps.synthesis_pass import GulpsDecompositionPass
from gulps.utils.invariants import GateInvariants

### Usage as a Decomposer

In [8]:
gate_set = [
    CXGate(),
    CXGate().power(1 / 2),
    iSwapGate().power(1 / 2),
    iSwapGate().power(1 / 3),
]
costs = [1.0, 1 / 2, 1 / 2, 1 / 3]
decomposer = GulpsDecomposer(gate_set, costs, precompute_polytopes=True)

In [9]:
N = 10_000
fidelities = []

for idx in tqdm(range(N)):
    u = random_unitary(4, seed=idx)  # Use different seeds
    v = Operator(decomposer(u))
    fid = average_gate_fidelity(u, v)
    fidelities.append(fid)

    if fid < 1 - 1e-6:
        print(f"Unitary {idx} fidelity is low: {fid:.8f}")
        print("Canonical invariants:")
        print("U:", c1c2c3(u))
        print("V:", c1c2c3(v))
        print("\n")
        continue

# Summary statistics
fidelities = np.array(fidelities)
print(f"\nSummary across {len(fidelities)} samples:")
print(f"  Median fidelity: {np.median(fidelities):.8f}")
print(f"  Mean fidelity:   {np.mean(fidelities):.8f}")
print(f"  Minimum fidelity:{np.min(fidelities):.8f}")

  0%|          | 0/10000 [00:00<?, ?it/s]

 86%|████████▌ | 8595/10000 [02:47<00:30, 46.39it/s]

Unitary 8589 fidelity is low: 0.99777063
Canonical invariants:
U: (np.float64(0.49997787), np.float64(0.16091816), np.float64(0.01681118))
V: (np.float64(0.5), np.float64(0.16091816), np.float64(0.01681113))




100%|██████████| 10000/10000 [03:15<00:00, 51.24it/s]


Summary across 10000 samples:
  Median fidelity: 1.00000000
  Mean fidelity:   0.99999978
  Minimum fidelity:0.99777063


In [10]:
# # %reload_ext  snakeviz
# %load_ext snakeviz
# import cProfile

# u = random_unitary(4, seed=0)
# cProfile.run("decomposer._run(u)", "profile_timings/temph.prof")